In [1]:
%cd ..

/Users/macos/Uni/1st_year/IntroDS/mini_project


In [317]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import darts
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from darts import TimeSeries
from darts.models import RegressionModel, ExponentialSmoothing
from darts.models.forecasting.arima import ARIMA
from sklearn.linear_model import Ridge
from sklearn.impute import KNNImputer
from tqdm.notebook import tqdm


In [3]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Open data

In [4]:
path = "res/global-data-on-sustainable-energy.csv"

In [5]:
df = pd.read_csv(path)

df.head()

,Entity,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Financial flows to developing countries (US $),Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),...,Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,Renewables (% equivalent primary energy),gdp_growth,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude
0,Afghanistan,2000,1.613591,6.2,9.22,20000.0,44.99,0.16,0.0,0.31,...,302.59482,1.64,760.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953
1,Afghanistan,2001,4.074574,7.2,8.86,130000.0,45.60,0.09,0.0,0.50,...,236.89185,1.74,730.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953
2,Afghanistan,2002,9.409158,8.2,8.47,3950000.0,37.83,0.13,0.0,0.56,...,210.86215,1.40,1029.999971,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953
3,Afghanistan,2003,14.738506,9.5,8.09,25970000.0,36.66,0.31,0.0,0.63,...,229.96822,1.40,1220.000029,NaN,8.832278,190.683814,60,652230.0,33.93911,67.709953
4,Afghanistan,2004,20.064968,10.9,7.75,NaN,44.24,0.33,0.0,0.56,...,204.23125,1.20,1029.999971,NaN,1.414118,211.382074,60,652230.0,33.93911,67.709953


In [256]:
df_nations = df[df['Entity'].isin(['Finland', 'France', 'Germany'])]

df_nations.head()

,Entity,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Financial flows to developing countries (US $),Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),...,Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,Renewables (% equivalent primary energy),gdp_growth,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude
1176,Finland,2000,100.0,100.0,NaN,NaN,31.68,23.93,22.48,23.38,...,69377.00,6.57,55100.00000,19.702833,5.773362,24285.46682,18,338145.0,61.92411,25.748151
1177,Finland,2001,100.0,100.0,NaN,NaN,29.25,29.98,22.77,21.54,...,70719.58,6.57,61090.00000,17.716715,2.610019,24946.18919,18,338145.0,61.92411,25.748151
1178,Finland,2002,100.0,100.0,NaN,NaN,29.52,32.60,22.30,19.83,...,70908.92,6.79,63439.99863,16.264904,1.707149,26869.67490,18,338145.0,61.92411,25.748151
1179,Finland,2003,100.0,100.0,NaN,NaN,28.35,42.29,22.73,19.06,...,75758.37,7.02,71709.99908,14.593681,2.003784,32855.13263,18,338145.0,61.92411,25.748151
1180,Finland,2004,100.0,100.0,NaN,NaN,31.10,37.25,22.72,25.63,...,75218.67,6.83,67680.00031,19.509205,3.992091,37702.84538,18,338145.0,61.92411,25.748151


In [259]:
trg_name = "Electricity from renewables (TWh)"

feat1 = 'Low-carbon electricity (% electricity)'
feat2 = 'Renewables (% equivalent primary energy)'
feat3 = 'Renewable energy share in the total final energy consumption (%)'
feat4 = 'gdp_per_capita'

In [257]:
trg_corr = df_nations.groupby('Year').mean(trg_name).corr()[trg_name]

trg_corr

Access to electricity (% of population)                                  NaN
Access to clean fuels for cooking                                        NaN
Renewable-electricity-generating-capacity-per-capita                     NaN
Financial flows to developing countries (US $)                           NaN
Renewable energy share in the total final energy consumption (%)    0.991043
Electricity from fossil fuels (TWh)                                -0.807720
Electricity from nuclear (TWh)                                     -0.953083
Electricity from renewables (TWh)                                   1.000000
Low-carbon electricity (% electricity)                              0.953837
Primary energy consumption per capita (kWh/person)                 -0.940048
Energy intensity level of primary energy (MJ/$2017 PPP GDP)        -0.960796
Value_co2_emissions_kt_by_country                                  -0.942229
Renewables (% equivalent primary energy)                            0.992621

# Try different models

## Preprocess

In [307]:
df_proc = df_nations \
            .copy() \
            [['Entity', 'Year', feat1, feat2,feat3, feat4, trg_name]]

df_proc.head()

,Entity,Year,Low-carbon electricity (% electricity),Renewables (% equivalent primary energy),Renewable energy share in the total final energy consumption (%),gdp_per_capita,Electricity from renewables (TWh)
1176,Finland,2000,65.711420,19.702833,31.68,24285.46682,23.38
1177,Finland,2001,59.644634,17.716715,29.25,24946.18919,21.54
1178,Finland,2002,56.376286,16.264904,29.52,26869.67490,19.83
1179,Finland,2003,49.702663,14.593681,28.35,32855.13263,19.06
1180,Finland,2004,56.483646,19.509205,31.10,37702.84538,25.63


In [265]:
def reverse_diff(s, start):
    s_undiff = [start]
    cum_sum = start
    for x in s:
        cum_sum += x
        s_undiff.append(cum_sum)

    # print(s_undiff)

    return np.array(s_undiff)

# reverse_diff(
#     scaler_trg.inverse_transform(df_proc['trg_diff_scaled'].iloc[1:].to_numpy().reshape(-1, 1)).squeeze(), 
#     df_proc[trg_name].iloc[0]
# ) - df_proc[trg_name]

### Do imputation

In [300]:
df_proc[df_proc[feat3].isna()]

,Entity,Year,Low-carbon electricity (% electricity),Renewables (% equivalent primary energy),Renewable energy share in the total final energy consumption (%),gdp_per_capita,Electricity from renewables (TWh),trg_diff,feat1_scaled,feat2_scaled,feat3_scaled,feat4_scaled
1196,Finland,2020,85.776360,33.168510,NaN,48744.98813,35.93,4.05,1.000000,1.0,NaN,0.671385
1217,France,2020,90.869606,14.788747,NaN,39030.36037,125.28,12.07,-0.046982,1.0,NaN,0.451133
1302,Germany,2020,55.681694,21.122828,NaN,46208.42947,251.48,11.15,1.000000,1.0,NaN,0.856147


In [315]:
period = 1

for nation in tqdm(df_proc['Entity'].unique()):
    df_nation = df_proc[df_proc['Entity'] == nation]

    # Do imputation
    X = df_nation[[feat1, feat2, feat3, feat4, trg_name]].to_numpy()
    Ximputed = KNNImputer(n_neighbors=2).fit_transform(X)

    df_proc.loc[df_nation.index, [feat1, feat2, feat3, feat4, trg_name]] = Ximputed
    
    # Make diff with target
    s_trg = df_proc[trg_name]
    s_trg_diff = s_trg.diff(period)


    df_proc.loc[df_nation.index, 'trg_diff'] = s_trg_diff
    
    # Scale (-1, 1) with feature
    df_proc.loc[df_nation.index, 'feat1_scaled'] = MinMaxScaler((-1, 1)).fit_transform(df_nation[feat1].to_numpy().reshape(-1, 1))
    df_proc.loc[df_nation.index, 'feat2_scaled'] = MinMaxScaler((-1, 1)).fit_transform(df_nation[feat2].to_numpy().reshape(-1, 1))
    df_proc.loc[df_nation.index, 'feat3_scaled'] = MinMaxScaler((-1, 1)).fit_transform(df_nation[feat3].to_numpy().reshape(-1, 1))
    df_proc.loc[df_nation.index, 'feat4_scaled'] = MinMaxScaler((-1, 1)).fit_transform(df_nation[feat4].to_numpy().reshape(-1, 1))

df_proc = df_proc.set_index(pd.to_datetime(df_proc['Year'], format='%Y'))

  0%|          | 0/3 [00:00<?, ?it/s]

In [316]:
df_proc.head()

,Entity,Year,Low-carbon electricity (% electricity),Renewables (% equivalent primary energy),Renewable energy share in the total final energy consumption (%),gdp_per_capita,Electricity from renewables (TWh),trg_diff,feat1_scaled,feat2_scaled,feat3_scaled,feat4_scaled
Year,,,,,,,,,,,,
2000-01-01,Finland,2000,65.711420,19.702833,31.68,24285.46682,23.38,NaN,-0.112442,-0.449884,-0.617461,-1.000000
2001-01-01,Finland,2001,59.644634,17.716715,29.25,24946.18919,21.54,-1.84,-0.448797,-0.663735,-0.896611,-0.954851
2002-01-01,Finland,2002,56.376286,16.264904,29.52,26869.67490,19.83,-1.71,-0.630001,-0.820055,-0.865594,-0.823414
2003-01-01,Finland,2003,49.702663,14.593681,28.35,32855.13263,19.06,-0.77,-1.000000,-1.000000,-1.000000,-0.414412
2004-01-01,Finland,2004,56.483646,19.509205,31.10,37702.84538,25.63,6.57,-0.624048,-0.470733,-0.684090,-0.083155


## Split train-test

In [318]:
df_proc_train, df_proc_test = df_proc[df_proc.index < '2015'].copy(), df_proc[df_proc.index >= '2015'].copy()

In [151]:
ts = darts.TimeSeries.from_series(df_proc_train.iloc[1:]['trg_diff'])
ts

<TimeSeries (DataArray) (Year: 14, component: 1, sample: 1)>
array([[[-1.84]],

       [[-1.71]],

       [[-0.77]],

       [[ 6.57]],

       [[-2.17]],

       [[-1.  ]],

       [[ 1.86]],

       [[ 3.45]],

       [[-6.07]],

       [[ 2.48]],

       [[ 0.  ]],

       [[ 4.38]],

       [[-2.93]],

       [[ 0.65]]])
Coordinates:
  * Year       (Year) datetime64[ns] 2001-01-01 2002-01-01 ... 2014-01-01
  * component  (component) object 'trg_diff'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [161]:
model = ExponentialSmoothing(seasonal_periods=2)

model.fit(ts2)
model.predict(10)

<TimeSeries (DataArray) (Year: 10, component: 1, sample: 1)>
array([[[25.88170881]],

       [[28.18157686]],

       [[26.64177974]],

       [[28.94164778]],

       [[27.40185066]],

       [[29.70171871]],

       [[28.16192159]],

       [[30.46178964]],

       [[28.92199252]],

       [[31.22186056]]])
Coordinates:
  * Year       (Year) datetime64[ns] 2015-01-01 2016-01-01 ... 2024-01-01
  * component  (component) object 'Electricity from renewables (TWh)'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [248]:
training_col = 'trg_diff'

# Construct future covariate
future_cov = darts.concatenate([
    darts.TimeSeries.from_series(df_proc.iloc[1:][feat1]),
    # darts.TimeSeries.from_series(df_proc.iloc[1:][feat2])
], axis=1)

# Construct target
ts = darts.TimeSeries.from_series(df_proc_train.iloc[1:][training_col])

# Fit model
model3 = RegressionModel(
    model=Ridge(),
    lags=2,
    lags_future_covariates=(0, 1),
)
model3.fit(ts, future_covariates=future_cov)

pred = model3.predict(6, future_covariates=future_cov).pd_dataframe()[training_col]

s_diff_pred = pd.concat([
    df_proc_train[training_col],
    pred
])

pred_rev = reverse_diff(s_diff_pred.iloc[1:], df_proc.iloc[0][trg_name])[15:]

mean_squared_error(
    df_proc.loc['2015':, trg_name],
    pred_rev
)

4.178501614276517

## Test with XGBoost

In [251]:
from darts.models.forecasting.xgboost import XGBModel

training_col = 'trg_diff'

# Construct future covariate
future_cov = concatenate([
    darts.TimeSeries.from_series(df_proc.iloc[1:][feat1]),
    darts.TimeSeries.from_series(df_proc.iloc[1:][feat2])
], axis=1)

# Construct target
ts = darts.TimeSeries.from_series(df_proc_train.iloc[1:][training_col])

# Fit model
model = XGBModel(
    lags=2,
    lags_future_covariates=(0, 1),
    # output_chunk_length=2
)
model.fit(ts, future_covariates=future_cov)

# make prediction
pred = model.predict(6, future_covariates=future_cov).pd_dataframe()[training_col]

s_diff_pred = pd.concat([
    df_proc_train[training_col],
    pred
])

pred_rev = reverse_diff(s_diff_pred.iloc[1:], df_proc.iloc[0][trg_name])[15:]

mean_squared_error(
    df_proc.loc['2015':, trg_name],
    pred_rev
)


9.011174813117197

## Test with static covariate

In [253]:
static_covs_multi = pd.DataFrame(data={"cont": [0, 2, 1], "cat": ["a", "c", "b"]})

series = TimeSeries.from_values(
    values=np.random.random((10, 3)),
    columns=["comp1", "comp2", "comp3"],
    static_covariates=static_covs_multi,
)
print(series.static_covariates)

static_covariates  cont cat
component                  
comp1               0.0   a
comp2               2.0   c
comp3               1.0   b
